## **Continual Learning - Replay Buffer**

In [ ]:
# Separate folders
EXEMPLAR_ROOT = "./UCF101/processed_exemplars"
SUBSET1_ROOT = "./UCF101/processed_subset1"

# PROCESS OLD CLASSES (The Memory Buffer)
# We limit 'train' to 20 samples per class to save space

limit = 20 
preprocess_dataset(
    target_classes=cfg.SELECTED_CLASSES, # The 10 original classes
    output_root=EXEMPLAR_ROOT,
    max_samples=limit
)

# PROCESS NEW CLASSES (Subset 1)
# We take 100% of the data for these new classes

preprocess_dataset_flexible(
    target_classes=cfg.TASK_1,
    output_root=SUBSET1_ROOT,
)

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1

# The "Master Map"
class_to_idx = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}

print(class_to_idx)

In [ ]:
batch_size = 8

# Load the 20 exemplars per class we preprocessed earlier
old_exemplars_dataset = UCF101Clips(
    "./UCF101/processed_exemplars/train",
    class_to_idx=class_to_idx
)

val_base = UCF101Clips("./UCF101/processed_exemplars/val", class_to_idx=class_to_idx)
val_sub1 = UCF101Clips("./UCF101/processed_subset1/val", class_to_idx=class_to_idx) # Added missing definition for val_sub1

test_base = UCF101Clips("./UCF101/processed_exemplars/test", class_to_idx=class_to_idx)
test_sub1 = UCF101Clips("./UCF101/processed_subset1/test", class_to_idx=class_to_idx)

# Load the full 10 classes of Subset 1
new_classes_dataset = UCF101Clips(
    "./UCF101/processed_subset1/train",
    class_to_idx=class_to_idx
)

# Merge: This dataset now contains labels 0-49 correctly mapped
continual_train_dataset = ConcatDataset([old_exemplars_dataset, new_classes_dataset])

load_continual_train = torch.utils.data.DataLoader(
    continual_train_dataset,
    batch_size=batch_size, # Use your consistent batch size
    shuffle=True,
    num_workers=2
)

load_combined_val_t1 = torch.utils.data.DataLoader(
    ConcatDataset([val_base, val_sub1]),
    batch_size=batch_size,
    shuffle=False
)

load_combined_test = torch.utils.data.DataLoader(
    ConcatDataset([test_base, test_sub1]),
    batch_size=batch_size,
    shuffle=False
)